# Mar 13/10 · 09:45 — TP v0.4 : inventaire CSV et ligne de commande

Mar 13/10, en deux parties :

| Horaire | Contenu |
|---|---|
| 10:00–10:25 | Partie 1 : inventaire CSV |
| 10:25–10:30 | Correction de la partie 1 |
| 10:45–11:10 | Partie 2 : ligne de commande |
| 11:10–11:15 | Correction de la partie 2, puis publication de la référence v0.4 |

Chaque correction commence à l'heure, partie finie ou non.

## Objectif

`servers.csv` remplace la liste `SERVERS` écrite dans le code. Chaque ligne fausse est notée avec son numéro de ligne, sans arrêter le script. Puis `fleetcheck` devient une commande : `--inventory`, `--env`, `--format`.

## Mode d'emploi du notebook

Celui de `tp/c02.ipynb` vaut toujours. Nouveau :

- `%run -n fleetcheck.py` charge les fonctions sans exécuter le bloc `if __name__ == "__main__":`. En partie 1, ce bloc n'existe pas encore : le script s'exécute en entier. En partie 2, il appelle `main()`, qui ne doit pas tourner dans le notebook (voir partie 2).
- `run(...)` accepte les arguments du script : `run("--env", "staging")` équivaut à `python3 fleetcheck.py --env staging`.
- Rendu : *Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `servers.csv`, `servers-broken.csv` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` de fin de TP v0.3, avec `config.json`. En retard : copiez `reference/fleetcheck.py` et `reference/config.json` du dépôt public, à la version `v0.3`.

## Partie 1 — inventaire CSV

Pendant cette partie, le fichier lu est fixé par une constante en tête de script :

```text
INVENTORY = "servers.csv"
```

La partie 2 la remplace par l'option `--inventory`.

### 1. Les deux inventaires — fin 10:01

Exécutez la cellule ci-dessous : elle écrit `servers.csv` et `servers-broken.csv` à côté de `fleetcheck.py` (les mêmes que `tp/` du dépôt public). Ouvrez-les et lisez l'en-tête.

In [ ]:
from pathlib import Path

Path("servers.csv").write_text("""name,env,ip,port,role
web-1,prod,127.0.0.1,8001,web
web-2,prod,127.0.0.1,8002,web
api-1,prod,127.0.0.1,8003,api
api-2,prod,127.0.0.1,8004,api
worker-1,prod,127.0.0.1,8005,worker
web-stg-1,staging,127.0.0.1,8006,web
api-stg-1,staging,127.0.0.1,8007,api
cache-1,prod,127.0.0.1,8009,cache
db-1,prod,192.0.2.1,5432,db
""", encoding="utf-8")
Path("servers-broken.csv").write_text("""name,env,ip,port,role
web-1,prod,127.0.0.1,8001,web
web-2,prod,127.0.0.1,8002,web
api-3,prod,127.0.0.1,abc,api
api-1,prod,127.0.0.1,8003,api
api-2,prod,127.0.0.1,8004,api
worker-1,prod,127.0.0.1,8005,worker
web-3,prod,300.0.0.1,8010,web
web-stg-1,staging,127.0.0.1,8006,web
api-stg-1,staging,127.0.0.1,8007,api
cache-1,prod,127.0.0.1,8009,cache
db-2,prod,192.0.2.2
db-1,prod,192.0.2.1,5432,db
""", encoding="utf-8")

Vérification : l'en-tête de `servers.csv` est `name,env,ip,port,role`, suivi de 9 lignes de données. `servers-broken.csv` contient les mêmes 9 serveurs plus 3 lignes fausses : port `abc` en ligne 4, IP `300.0.0.1` en ligne 8, ligne à 3 champs en ligne 12.

In [ ]:
with open("servers.csv", encoding="utf-8") as f:
    lignes = f.read().splitlines()
check((lignes[0], len(lignes) - 1), ("name,env,ip,port,role", 9))
with open("servers-broken.csv", encoding="utf-8") as f:
    lignes = f.read().splitlines()
check((lignes[3], lignes[7], lignes[11]),
      ("api-3,prod,127.0.0.1,abc,api", "web-3,prod,300.0.0.1,8010,web", "db-2,prod,192.0.2.2"))

### 2. `load(path)` — fin 10:20

`load(path)` remplace la liste `SERVERS` et la `load` de v0.2. Elle renvoie toujours `servers, invalid`.

Chaque ligne est numérotée comme dans l'éditeur : l'en-tête est la ligne 1. `HEADER` est la liste des 5 colonnes ; `validate` est celle de v0.2, **sans la modifier**.

```mermaid
flowchart TD
  lire(["fichier lisible ?"]) -- non --> fatal["erreur fatale, étape 3"]
  lire -- oui --> entete(["en-tête : name,env,ip,port,role ?"])
  entete -- non --> fatal
  entete -- oui --> ligne["ligne suivante"]
  ligne --> vide(["ligne vide ?"])
  vide -- oui --> reste
  vide -- non --> champs(["5 champs ?"])
  champs -- non --> invalid["ajoutée à invalid"]
  champs -- oui --> valide(["validate(dict(zip(HEADER, row))) lève InvalidServer ?"])
  valide -- oui --> invalid
  valide -- non --> servers["ajoutée à servers"]
  invalid --> reste(["reste des lignes ?"])
  servers --> reste
  reste -- oui --> ligne
  reste -- non --> aucune(["servers vide ?"])
  aucune -- oui --> fatal
  aucune -- non --> fin["return servers, invalid"]
```

Chaque ligne invalide devient un dict au format suivant :

```text
{"line": 4, "raw": "api-3,prod,127.0.0.1,abc,api", "error": "port non entier : 'abc'"}
```

`raw` est la ligne telle qu'elle est dans le fichier. Une ligne à 3 champs a pour erreur `3 champs au lieu de 5`.

Le résumé texte cite désormais la ligne et non plus le nom (une ligne à 3 champs n'a pas de nom fiable) : `invalide ligne <n> : <erreur>`. `report.json` suit : `invalid_lines` passe au format ci-dessus.

**Prédire** — Un collègue ouvre `servers.csv` dans Excel et l'enregistre en « CSV UTF-8 » : le fichier commence désormais par un BOM, le caractère invisible `\ufeff`. Que fait votre `load`, qui lit en `encoding="utf-8"` et vérifie l'en-tête ?

In [ ]:
# Ma réponse :

Essayez ici, par exemple `load("servers-broken.csv")` :

In [ ]:
%run -n fleetcheck.py


Vérification de `load` sur les deux inventaires :

In [ ]:
%run -n fleetcheck.py
servers, invalid = load("servers.csv")
check((len(servers), invalid), (9, []))
check(servers[0], {"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": 8001, "role": "web"})
servers, invalid = load("servers-broken.csv")
check(len(servers), 9)
check(invalid, [
    {"line": 4, "raw": "api-3,prod,127.0.0.1,abc,api", "error": "port non entier : 'abc'"},
    {"line": 8, "raw": "web-3,prod,300.0.0.1,8010,web", "error": "IP invalide : '300.0.0.1'"},
    {"line": 12, "raw": "db-2,prod,192.0.2.2", "error": "3 champs au lieu de 5"},
])

Vérification du script avec `INVENTORY = "servers.csv"` :

In [ ]:
r = run()
check(r.stdout, """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
""")

« invalides » reste au pluriel à 0 : c'est le comportement de la référence.

Avec `INVENTORY = "servers-broken.csv"`, la sortie attendue est la suivante. Pour la voir, changez la constante, enregistrez, exécutez `run()` dans la cellule d'essai ci-dessous, puis remettez `"servers.csv"`. Elle est vérifiée en fin de partie 2, avec `--inventory`.

```text
9 serveurs valides, 3 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
invalide ligne 4 : port non entier : 'abc'
invalide ligne 8 : IP invalide : '300.0.0.1'
invalide ligne 12 : 3 champs au lieu de 5
```

In [ ]:
r = run()


### 3. Cas fatals — fin 10:25

Trois cas arrêtent le script avec un message d'une ligne et `sys.exit(1)`, comme `load_config` :

| Cas | Message |
|---|---|
| Fichier absent | `inventaire illisible : <détail de l'erreur>` |
| En-tête absent ou faux | `en-tête attendu dans <fichier> : name,env,ip,port,role` |
| Aucune ligne valide | `aucune ligne valide dans <fichier>` |

**Prédire** — Un `servers.csv` vide (0 octet) : lequel des trois cas s'applique ? Votre `load` le traite-t-elle sans trace Python ?

In [ ]:
# Ma réponse :

Vérification : la cellule appelle `load` sur un fichier absent, puis sur deux fichiers qu'elle crée et supprime ensuite. `arret` renvoie ce que `load` a affiché et le code passé à `sys.exit`.

In [ ]:
%run -n fleetcheck.py
import contextlib
import io
from pathlib import Path

def arret(path):
    sortie = io.StringIO()
    with contextlib.redirect_stdout(sortie):
        try:
            load(path)
        except SystemExit as e:
            return sortie.getvalue(), e.code
    return sortie.getvalue(), "pas de sys.exit"

Path("entete-faux.csv").write_text("nom,env,ip,port,role\nweb-1,prod,127.0.0.1,8001,web\n", encoding="utf-8")
Path("aucune-valide.csv").write_text("name,env,ip,port,role\napi-3,prod,127.0.0.1,abc,api\n", encoding="utf-8")
try:
    check(arret("absent.csv"), ("inventaire illisible : [Errno 2] No such file or directory: 'absent.csv'\n", 1))
    check(arret("entete-faux.csv"), ("en-tête attendu dans entete-faux.csv : name,env,ip,port,role\n", 1))
    check(arret("aucune-valide.csv"), ("aucune ligne valide dans aucune-valide.csv\n", 1))
finally:
    Path("entete-faux.csv").unlink()
    Path("aucune-valide.csv").unlink()

Si votre `load` ne renvoie pas les 9 valides de `servers.csv` à 10:15, terminez l'étape 2 jusqu'à 10:25 ; l'étape 3 se fait alors à la maison, à partir de la référence.

### Avant 10:45

La `load` corrigée est publiée à 10:25 dans `tp/c05/load.py` du dépôt public. Si la vôtre ne passe pas les vérifications ci-dessus, collez ce fichier dans `fleetcheck.py` à la place de votre `load` avant 10:45 : la partie 2 en dépend. Remplacez aussi la ligne des invalides du résumé par celle indiquée en fin de fichier : l'ancienne lit `line['name']`, qui n'existe plus.

### À éviter

- Ouvrir le CSV sans `newline=""` : lignes vides intercalées sous Windows à l'écriture.
- Compter les lignes à partir de 0, ou sans l'en-tête : le numéro ne correspond plus à l'éditeur.
- Recopier `validate` au lieu de la réutiliser.

## Partie 2 — ligne de commande

`main()` lit ses options dans `sys.argv`. Dans le notebook, `sys.argv` contient les arguments du noyau Jupyter (`-f …kernel.json`) : appeler `main()` ou exécuter `%run fleetcheck.py` sans `-n` fait échouer argparse. Les fonctions se vérifient après `%run -n`, la ligne de commande avec `run(...)`.

### 1. `main()` et le parser — fin 10:55

Écrivez `main()` avec un parser `argparse` à 3 options :

| Option | Valeur |
|---|---|
| `--inventory` | fichier CSV, défaut `servers.csv` |
| `--env` | facultatif, un environnement |
| `--format` | `text` (défaut) ou `json`, rien d'autre |

Chaque option a un `help`. La constante `INVENTORY` disparaît. Le script appelle `main()` sous `if __name__ == "__main__":`.

Vérification : l'aide liste `--inventory`, `--env` et `--format`, et sort en `0`.

In [ ]:
r = run("--help")
check([option in r.stdout for option in ("--inventory", "--env", "--format")], [True, True, True])
check(r.returncode, 0)

### 2. `--env` — fin 11:02

`--env` filtre les serveurs valides avant le résumé et le rapport. Sans `--env`, rien ne change.

Vérification : la dernière ligne s'arrête après `prod/web : ` (espace final compris) : aucun serveur `prod` ne reste après le filtre.

In [ ]:
r = run("--env", "staging")
check(r.stdout, """2 serveurs valides, 0 invalides
par env : staging 2
par rôle : web 1, api 1
prod/web : 
""")

### 3. `--format json` — fin 11:10

`--format json` affiche le contenu du rapport sur stdout, avec `json.dumps(report, indent=2, ensure_ascii=False)`. `--format text` garde le résumé. `report.json` est écrit dans les deux cas.

Vérification : la sortie standard se relit avec `json.loads`, et c'est le contenu de `report.json`.

In [ ]:
import json

r = run("--format", "json")
report = json.loads(r.stdout)
check(report["summary"], {"total": 9, "invalid": 0})
with open("report.json", encoding="utf-8") as f:
    check(json.load(f), report)
check(r.returncode, 0)

**Prédire** — `run("--format", "xml")` : votre code s'exécute-t-il ? Quel code de sortie, et sur quel flux arrive le message ?

In [ ]:
# Ma réponse :

Un format inconnu : argparse affiche l'usage puis une erreur sur stderr, et sort en `2` avant que votre code ne tourne. Dernière ligne attendue, avec Python 3.14 :

```text
fleetcheck.py: error: argument --format: invalid choice: 'xml' (choose from text, json)
```

In [ ]:
r = run("--format", "xml")
check(r.stderr.splitlines()[-1], "fleetcheck.py: error: argument --format: invalid choice: 'xml' (choose from text, json)")
check(r.returncode, 2)

Enfin, `--inventory servers-broken.csv` redonne la sortie attendue de la partie 1. C'est le critère de fin du TP.

In [ ]:
r = run("--inventory", "servers-broken.csv")
check(r.stdout, """9 serveurs valides, 3 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
invalide ligne 4 : port non entier : 'abc'
invalide ligne 8 : IP invalide : '300.0.0.1'
invalide ligne 12 : 3 champs au lieu de 5
""")

### À observer

**Prédire** — `--env prd` est une faute de frappe pour `prod`. Que va afficher `run("--env", "prd")`, avec quel code de sortie ? Un job de CI qui lance cette commande finit-il vert ou rouge, et est-ce le bon résultat ?

In [ ]:
# Ma réponse :

In [ ]:
r = run("--env", "prd")

`--env prd` affiche `0 serveurs valides, 0 invalides` et sort en `0` : une CI serait verte sans avoir rien contrôlé. v0.5 corrige ce défaut.

## Pour aller plus loin

Régénérez `servers.csv` avec le serveur simulé : `simserver.py` du dépôt public copié à côté de `fleetcheck.py`, puis dans un terminal de JupyterLab `python3 simserver.py --inventory > servers.csv` (`py` sous Windows). Le contenu doit être identique à `tp/servers.csv`. Sous Windows PowerShell 5, `>` écrit de l'UTF-16 : la lecture lève alors `UnicodeDecodeError`, et `tp/servers.csv` reste la bonne copie.

### Une ligne de log : l'expression régulière

Un CSV se lit par `csv`, un JSON par `json`, une IP par `ipaddress`, une ligne à séparateur fixe par `split`. Une ligne de log au format libre, comme celle d'un nginx, n'a aucun de ces outils : on la décrit par une **expression régulière** (module `re`), et chaque morceau utile devient un **groupe nommé**.

- `r"..."` : chaîne brute, `\d` arrive tel quel au moteur de regex. Sans le `r`, Python 3.14 avertit : `"\d" is an invalid escape sequence`.
- `re.search` trouve le motif n'importe où dans la ligne ; `re.fullmatch` exige que **toute** la chaîne corresponde : c'est lui qui valide un champ.
- `(?P<status>\d{3})` capture trois chiffres sous le nom `status`, lus par `m["status"]` : une chaîne, pas un entier.
- Pas de correspondance : `None`. `m["status"]` lève alors `TypeError: 'NoneType' object is not subscriptable` : tester `if m is None` avant de lire.

In [ ]:
import re

line = '198.51.100.23 - - [20/Nov/2026:02:07:14 +0100] "POST /login HTTP/1.1" 502 157 "-" "curl/8.4.0" 30.001'
ACCESS = r'"(?P<method>[A-Z]+) (?P<path>\S+) [^"]*" (?P<status>\d{3}) '
m = re.search(ACCESS, line)
check((m["method"], m["path"], m["status"]), ("POST", "/login", "502"))
check(re.search(ACCESS, "198.51.100.23 - - [20/Nov/2026:02:07"), None)
check((re.fullmatch(r"\d{3}", "5021"), re.search(r"\d{3}", "5021") is not None), (None, True))

À vous : une fonction `parse_access(line)` qui renvoie `{"method": …, "path": …, "status": 502, "latency_s": 30.001}` (le dernier champ est la durée de la requête), ou lève `ValueError` si la ligne ne correspond pas. Essayez-la sur une ligne tronquée. C'est le cœur du projet « analyseur de logs ».

## En retard

La version de référence v0.4 (`fleetcheck.py`, `config.json`, `servers.csv`, `servers-broken.csv`) est publiée dans `reference/` à 11:15. Copiez-la dans votre dépôt pour démarrer le TP v0.5.